# Retail medallion gold aggregations

This notebook publishes Direct Lake-ready gold tables that align exactly with the RetailSales TMDL source columns.
It preserves coherent Date, Customer, Product, and Sales shapes from the committed datasets.


In [ ]:
ENVIRONMENT = "dev"
PROCESSING_DATE = "2026-03-02"
PIPELINE_RUN_ID = "local-demo"
environment = ENVIRONMENT
processing_date = PROCESSING_DATE
pipeline_run_id = PIPELINE_RUN_ID


In [ ]:
from pyspark.sql import functions as F
import json

customers_silver_table = 'silver.retail_customers'
products_silver_table = 'silver.retail_products'
sales_silver_table = 'silver.retail_sales_enriched'

gold_date_table = 'gold.Date'
gold_customer_table = 'gold.Customer'
gold_product_table = 'gold.Product'
gold_sales_table = 'gold.Sales'
gold_summary_table = 'gold.sales_monthly_summary'

gold_date_columns = ['date_key', 'date_value', 'calendar_year', 'calendar_quarter', 'month_number', 'month_name', 'year_month']
gold_customer_columns = ['customer_key', 'customer_id', 'customer_name', 'loyalty_tier', 'home_region', 'signup_date', 'email_domain', 'synthetic_record']
gold_product_columns = ['product_key', 'product_id', 'product_name', 'category', 'subcategory', 'unit_price', 'unit_cost', 'is_active', 'synthetic_record']
gold_sales_columns = ['sales_key', 'date_key', 'customer_key', 'product_key', 'purchase_id', 'purchase_ts', 'ingest_ts', 'store_id', 'channel', 'quantity', 'unit_price', 'unit_cost', 'revenue', 'cost', 'gross_margin', 'gross_margin_pct', 'anomaly_label', 'synthetic_record']

spark.conf.set('spark.sql.parquet.vorder.default', 'true')
spark.conf.set('spark.databricks.delta.optimizeWrite.enabled', 'true')
spark.conf.set('spark.databricks.delta.optimizeWrite.binSize', '1g')


In [ ]:
def log_event(stage, message, **metrics):
    record = {
        'stage': stage,
        'message': message,
        'environment': environment,
        'processing_date': processing_date,
        'pipeline_run_id': pipeline_run_id,
        'metrics': metrics,
    }
    print(json.dumps(record, sort_keys=True))

cutoff_date = F.to_date(F.lit(processing_date))
sales_base_df = spark.table(sales_silver_table).filter(F.col('purchase_date') <= cutoff_date)
customers_base_df = spark.table(customers_silver_table)
products_base_df = spark.table(products_silver_table)
spark.sql('CREATE SCHEMA IF NOT EXISTS gold')


In [ ]:
date_df = (
    sales_base_df
    .select('purchase_date')
    .distinct()
    .withColumn('date_key', F.date_format('purchase_date', 'yyyyMMdd').cast('long'))
    .withColumnRenamed('purchase_date', 'date_value')
    .withColumn('calendar_year', F.year('date_value'))
    .withColumn('calendar_quarter', F.concat(F.lit('Q'), F.quarter('date_value')))
    .withColumn('month_number', F.month('date_value'))
    .withColumn('month_name', F.date_format('date_value', 'MMMM'))
    .withColumn('year_month', F.date_format('date_value', 'yyyy-MM'))
    .select(*gold_date_columns)
)

customer_df = (
    customers_base_df
    .select(*gold_customer_columns)
    .dropDuplicates(['customer_key'])
)

product_df = (
    products_base_df
    .select(*gold_product_columns)
    .dropDuplicates(['product_key'])
)

sales_df = (
    sales_base_df
    .withColumn('date_key', F.date_format('purchase_date', 'yyyyMMdd').cast('long'))
    .select(*gold_sales_columns)
)

summary_df = (
    sales_base_df
    .groupBy('year_month', 'channel')
    .agg(
        F.countDistinct('purchase_id').alias('purchase_count'),
        F.sum('quantity').alias('total_quantity'),
        F.round(F.sum('revenue'), 2).alias('total_revenue'),
        F.round(F.sum('cost'), 2).alias('total_cost'),
        F.round(F.sum('gross_margin'), 2).alias('total_gross_margin'),
    )
    .withColumn('gross_margin_pct', F.round(F.when(F.col('total_revenue') != 0, F.col('total_gross_margin') / F.col('total_revenue')), 6))
)


In [ ]:
date_df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(gold_date_table)
customer_df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(gold_customer_table)
product_df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(gold_product_table)
(
    sales_df.write.format('delta')
    .mode('overwrite')
    .partitionBy('date_key')
    .option('overwriteSchema', 'true')
    .saveAsTable(gold_sales_table)
)
(
    summary_df.write.format('delta')
    .mode('overwrite')
    .partitionBy('year_month')
    .option('overwriteSchema', 'true')
    .saveAsTable(gold_summary_table)
)

spark.sql(f'OPTIMIZE {gold_sales_table} ZORDER BY (date_key, customer_key, product_key)')
spark.sql(f'OPTIMIZE {gold_summary_table} ZORDER BY (year_month, channel)')

log_event(
    'gold_validation',
    'Retail gold publishing complete',
    date_rows=date_df.count(),
    customer_rows=customer_df.count(),
    product_rows=product_df.count(),
    sales_rows=sales_df.count(),
    summary_rows=summary_df.count(),
)
spark.sql(f'SELECT COUNT(*) AS gold_sales_rows FROM {gold_sales_table}').show()
spark.sql(f'SELECT COUNT(*) AS gold_customer_rows FROM {gold_customer_table}').show()
spark.sql(f'SELECT COUNT(*) AS gold_product_rows FROM {gold_product_table}').show()
spark.sql(f'SELECT COUNT(*) AS gold_date_rows FROM {gold_date_table}').show()
